## Reference Validation

- Parameter ranges based on Liu, Oosterlee and Bohte (2019). 
- The main pricing reference is QuantLib AnalyticalHestonEngine.
- In order to complete a cross-check, an independently coded Carr-Madan Fourier was created.

In [30]:
#uncomment for library installation
%pip -q QuantLib numpy pandas scikit-learn matplotlib joblib

ERROR: unknown command "QuantLib"
Note: you may need to restart the kernel to use updated packages.


In [31]:
#libraries
from __future__ import annotations 
import json 
import platform 
import warnings 
from pathlib import Path 
from time import perf_counter 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd 
import QuantLib as ql 
import scipy 
from scipy.integrate import IntegrationWarning, quad 
from scipy.stats import qmc

In [32]:
#config
SEED = 42 
N_SAMPLES = 100 
N_STRIKE_CURVES = N_SAMPLES
N_INTERDEPENDENT_CASES = 24 # We utilize 12 valid Feller tests/12 violating feller tests
K_FIXED = 1.0 
DIVIDEND_YIELD = 0.0 
VALUATION_DATE = ql.Date(2, ql.January, 2026)
DAY_COUNT = ql.Actual365Fixed()

INTEGRATION_ORDERS = (32, 64, 128)
REFERENCE_ORDER = 128 
CONVERGENCE_TOL = 1e-5 
INDEPENDENT_TOL = 1e-4 
BOUND_TOL = 1e-8 
SHAPE_TOL = 1e-6 
PARITY_TOL = 1e-6 
BENCHMARK_REPEAT = 5 
FOURIER_UPPER_LIMIT = 200.0 

In [33]:
OUTPUT = Path("reference_validation")
OUTPUT.mkdir(parents=True, exist_ok=True)
ql.Settings.instance().evaluationDate = VALUATION_DATE

In [34]:
#Heston Pricing Model 
def heston_quantlib_prices(
        S0,
        K,
        T,
        r,
        v0,
        kappa,
        theta,
        sigma,
        rho,
        q=DIVIDEND_YIELD, 
        integration_order=REFERENCE_ORDER, 
        option_type = ql.Option.Call
):
    ql.Settings.instance().evaluationDate = VALUATION_DATE

    maturity = (
        VALUATION_DATE
        + max(1,int(round(float(T) * 365)))
    )

    spot = ql.QuoteHandle(
        ql.SimpleQuote(float(S0))
    )

    risk_free_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(
            VALUATION_DATE,
            float(r),
            DAY_COUNT
        )
    )

    dividend_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(
            VALUATION_DATE,
            float(q),
            DAY_COUNT
        )
    )

    process = ql.HestonProcess(
        risk_free_curve,
        dividend_curve,
        spot,
        float(v0),
        float(kappa),
        float(theta),
        float(sigma),
        float(rho)
    )

    model = ql.HestonModel(process)
    engine = ql.AnalyticHestonEngine(model, int(integration_order))
    payoff = ql.PlainVanillaPayoff(
        option_type,
        float(K)
    )
    exercise = ql.EuropeanExercise(maturity)
    option = ql.VanillaOption(
        payoff,
        exercise
    )
    option.setPricingEngine(engine)

    return float(option.NPV())

In [35]:
def pricing_kwargs(row): 
    return dict(
        S0 = float(row.S0), 
        K = float(row.K), 
        T = float(row.T),
        r = float(row.r),
        v0 = float(row.v0),
        kappa = float(row.kappa),
        theta = float(row.theta), 
        sigma = float(row.sigma),
        rho = float(row.rho),
        q = float(row.q),
    )

In [36]:
def sample_liu_params(n, seed):
    columns = [
        "m", 
        "T", 
        "r", 
        "rho", 
        "kappa", 
        "theta", 
        "sigma", 
        "v0"
    ]
    lower = [0.6, 0.1, 0.0, -0.95, 0.0, 0.0, 0.0, 0.05]
    upper = [1.4, 1.4, 0.1, 0.0, 2.0, 0.5, 0.5, 0.5]
    
    sampler = qmc.LatinHypercube(d=len(columns), seed=seed)
    values = qmc.scale(sampler.random(n), lower, upper)
    frame = pd.DataFrame(values, columns=columns)
    frame.insert(0, "sample_id", np.arange(n))
    
    frame["K"] = K_FIXED
    frame["S0"] = frame["m"] * frame["K"]
    frame["q"] = DIVIDEND_YIELD 
    frame["T_effective"] = np.maximum(
        1,
        np.round(frame["T"]* 365).astype(int)
    )/365.0

    frame["feller_satisfied"] = (
        2 * frame["kappa"] * frame["theta"] 
        >= frame["sigma"] ** 2 
    )
    return frame 

In [37]:
def run_reference_checks(frame):
    records = []
    for i, row in enumerate(frame.itertuples(index=False),start=1):
        record = row._asdict()
        params = pricing_kwargs(row)
        for order in INTEGRATION_ORDERS:
            start = perf_counter()
            try:
                price = heston_quantlib_prices(
                    **params, 
                    integration_order=order
                )
                error = ""
            except Exception as exc: 
                price = np.nan
                error = f"{type(exc).__name__}: {exc}"
            record[f"price_{order}"] = price 
            record[f"seconds_{order}"] = perf_counter() - start
            record[f"error_{order}"] = error 
        p32, p64, p128 = (record[f"price_{n}"] for n in (32, 64, 128))
        record["difference_32_64"] = abs(p32 - p64)
        record["difference_64_128"] = abs(p64 - p128)
        record["converged_32_64"] = bool(
            np.isfinite(record["difference_32_64"])
            and record["difference_32_64"]<= CONVERGENCE_TOL
        )
        record["converged_64_128"] = bool(
            np.isfinite(record["difference_64_128"])
            and record["difference_64_128"]<= CONVERGENCE_TOL
        )

        t = row.T_effective
        lower = max(
            0.0, 
            row.S0 * np.exp(-row.q * t) 
            - row.K * np.exp(-row.r * t)
        )
        upper = row.S0 * np.exp(-row.q * t)
        record["lower_bound"] = lower 
        record["upper_bound"] = upper 
        record["valid_price"] = bool(
            np.isfinite(p128)
            and lower - BOUND_TOL
            <= p128
            <= upper + BOUND_TOL
        )
        records.append(record)
        if i % 20 == 0 or i == len(frame):
            print(
                f"Reference pricing: {i}/{len(frame)} parameter combinations"
            )
    return pd.DataFrame(records)

In [38]:
# Financial checks on 100 strike curves
def run_shape_checks(frame):
    records = []
    for row in frame.head(N_STRIKE_CURVES).itertuples(index = False):
        params = pricing_kwargs(row)
        moneyness = np.linspace(0.6, 1.4, 17)
        strikes = np.sort(row.S0/moneyness)
        curve_error, parity_error_message = "", ""
        monotone, convex, parity_error = None, None, np.nan
        try:
            calls = np.asarray([
                heston_quantlib_prices(**{**params, "K": float(k)},
                                      integration_order=REFERENCE_ORDER)
                for k in strikes
            ])
            if not np.isfinite(calls).all():
                raise ValueError("Nonfinite call found in strike curve")
            monotone = bool(np.all(np.diff(calls) <= SHAPE_TOL))
            slopes = (np.diff(calls)/np.diff(strikes))
            convex = bool(np.all(np.diff(slopes) >= -SHAPE_TOL))
        except Exception as exc: 
            curve_error = f"{type(exc).__name__}: {exc}"
        
        #separate try: put/call failure that must not invalidate strike tests

        try:
            call = heston_quantlib_prices(
                **params,
                integration_order=REFERENCE_ORDER, 
                option_type=ql.Option.Call
            )
            put = heston_quantlib_prices(
                **params, 
                integration_order=REFERENCE_ORDER, 
                option_type=ql.Option.Put
            )
            t = row.T_effective 
            parity_rhs = row.S0 * np.exp(-row.q * t) - row.K * np.exp(-row.r * t)
            parity_error = float(abs(call - put - parity_rhs))

        except Exception as exc:
            parity_error_message = f"{type(exc).__name__}: {exc}"
        
        records.append({
            "sample_id": int(row.sample_id), 
            "feller_satisfied": bool(row.feller_satisfied),
            "strike_monotonicity_ok": monotone, 
            "strike_convexity_ok": convex, 
            "put_call_parity_abs_error": parity_error, 
            "put_call_parity_ok": bool(parity_error <= PARITY_TOL) 
            if np.isfinite(parity_error) else None, 
            "strike_curve_error": curve_error, 
            "parity_error_message": parity_error_message,
        })
    return pd.DataFrame(records)

In [39]:
# Independent Fourier Solver (Carr-Madan)
def heston_characteristic_function(u, *, S0, T, r, q, v0, kappa, theta, sigma, rho):
    iu = 1j * u 
    b = kappa - rho * sigma * iu 
    d = np.sqrt(b * b + sigma * sigma * (u * u + iu))
    g = (b - d)/ (b + d)
    exp_dt = np.exp(-d * T)
    C = (iu * (np.log(S0) + (r-q)*T)
         + (kappa * theta / sigma ** 2)
         * ((b-d) * T - 2 * np.log((1 - g * exp_dt) / (1-g))))
    D = ((b-d)/sigma **2) * (1 - exp_dt)/(1-g * exp_dt)
    return np.exp(C + D * v0)

def heston_fourier_call(*, S0, K, T, r, q, v0, kappa, theta, sigma, rho, alpha=1.0, upper=FOURIER_UPPER_LIMIT):
    if sigma <= 0: 
        raise ValueError("Fourier formula needs sigma > 0")
    k = np.log(K)
    cf_params = dict(S0=S0, T=T, r=r, q=q, v0=v0, kappa=kappa, theta=theta, sigma=sigma, rho=rho)

    def integrand(u):
        z = complex(u) - 1j * (alpha + 1.0)
        phi = heston_characteristic_function(z, **cf_params)
        denominator = (alpha + 1j * u) * (alpha + 1 + 1j * u)
        return float(np.real(np.exp(-1j * u * k) * np.exp(-r * T) * phi / denominator))
    
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", IntegrationWarning)
        integral, estimated_error = quad(
            integrand, 0.0, upper, epsabs=1e-10, epsrel=1e-9, limit=500
        )
    
    price = float(np.exp(-alpha * k) * integral / np.pi)
    if not np.isfinite(price):
        raise ArithmeticError("Fourier integral returned nonfinite price")
    messages = "; ".join(str(w.message) for w in caught)
    return price, float(estimated_error), messages

def run_independent_check(frame, reference_resulst): 
    #including both Feller-valid and Feller-violating cases when available:
    n_bad = N_INTERDEPENDENT_CASES//2
    bad = frame.loc[~frame.feller_satisfied].head(n_bad)
    good = frame.loc[frame.feller_satisfied].head(N_INTERDEPENDENT_CASES - len(bad))
    selected = pd.concat([bad, good], ignore_index=True)
    ref_by_id = reference_resulst.set_index("sample_id")
    rows = []

    for row in selected.itertuples(index=False):
        ql_price = float(ref_by_id.loc[row.sample_id, f"price_{REFERENCE_ORDER}"])
        try:
            fourier_price, quad_error, warning_message = heston_fourier_call(
                S0=float(row.S0), 
                K=float(row.K), 
                T=float(row.T_effective), 
                r=float(row.r),
                q=float(row.q),
                v0=float(row.v0),
                kappa=float(row.kappa),
                theta=float(row.theta),
                sigma=float(row.sigma),
                rho=float(row.rho)
            )
            error = ""
        except Exception as exc:
            fourier_price, quad_error, warning_message = np.nan, np.nan, ""
            error = f"{type(exc).__name__}: {exc}"
        abs_diff = abs(ql_price - fourier_price)
        rows.append({
            "sample_id":int(row.sample_id),
            "feller_satisfied": bool(row.feller_satisfied),
            "quantlib_price": ql_price,
            "fourier_price": fourier_price,
            "abs_difference": abs_diff, 
            "within_tolerance": bool(np.isfinite(abs_diff) and abs_diff <= INDEPENDENT_TOL),
            "quadrature_error_estimate": quad_error, 
            "fourier_warning": warning_message, 
            "error": error
        })
    return pd.DataFrame(rows)

In [40]:
def benchmark_integration_orders(frame):
    records = []
    rows = list(frame.itertuples(index=False))
    rng = np.random.default_rng(SEED)
    for order in INTEGRATION_ORDERS:
        heston_quantlib_prices(**pricing_kwargs(rows[0]), integration_order=order)
    for repeat in range(BENCHMARK_REPEAT): 
        for order in rng.permutation(INTEGRATION_ORDERS):
            start = perf_counter()
            for row in rows: 
                heston_quantlib_prices(**pricing_kwargs(row), integration_order=int(order))
            elapsed = perf_counter()-start
            records.append({
                "repeat": repeat + 1, 
                "integration_order": int(order), 
                "n_option": len(rows), 
                "total_seconds": elapsed, 
                "millisecond_per_option": 1000.0 * elapsed / len(rows),
            })
    trials = pd.DataFrame(records)
    summary = trials.groupby("integration_order", as_index=False).agg(
        median_ms_per_option=("millisecond_per_option", "median"),
        min_ms_per_option = ("millisecond_per_option", "min"),
        max_ms_per_option=("millisecond_per_option", "max")
    )
    return trials, summary

In [41]:
def plot_convergence(results):
    fig, ax = plt.subplots(figsize=(10, 10))
    rng = np.random.default_rng(SEED)
    comparison = [("32 vs 64", "difference_32_64"),("64 vs 128", "difference_64_128")]
    any_positive = False
    for i, (label, col) in enumerate(comparison, start=1): 
        vals = results[col].dropna().to_numpy(dtype=float)
        positive = vals[vals > 0]
        n_zeros = int(np.sum(vals==0))
        if len(positive):
            any_positive = True 
            x = i + rng.uniform(-0.13, 0.13, len(positive))
            ax.scatter(x, positive, s=21, alpha=0.55)
        max_text = f"{np.max(vals):.2e}" if len(vals) else "n/a"
        ax.text(i, 0.98, f"Maximum: {max_text}\nExact zeroes: {n_zeros}",
                ha="center", va="top", transform=ax.get_xaxis_transform(), fontsize=9)
    if any_positive:
        ax.set_yscale("log")
    else:
        ax.text(0.5, 0.5, "All difference are numerically zero", transform=ax.transAxes, ha="center")
    
    ax.set_xticks([1,2])
    ax.set_xticklabels([label for label, _ in comparison])
    ax.set_xlim(0.55, 2.45)
    ax.set_xlabel("Integration-order comparison")
    ax.set_ylabel("Absolute price differnce (log scale; zeros excluded)")
    ax.set_title("Heston numerical integration stability")
    ax.grid(axis="y", alpha=0.25)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUTPUT/"integration_stability.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_benchmark(timing_summary):
    fig, ax = plt.subplots(figsize=(10,10))
    x = timing_summary["integration_order"].to_numpy()
    y = timing_summary["median_ms_per_option"].to_numpy()
    ax.plot(x, y, marker="o", linewidth=2)
    for xi, yi, in zip(x,y):
        ax.annotate(f"{yi:.3f} ms", (xi, yi), xytext=(0,9), textcoords="offset points", ha="center", fontsize=9)
    ax.set_xticks(list(INTEGRATION_ORDERS))
    ax.set_xlabel("QuantLib integration order")
    ax.set_ylabel("Median milliseconds per option")
    ax.set_title("Heston Pricing cost by integration order")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.25)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUTPUT/"integration_timing.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


In [42]:
def main():
    frame = sample_liu_params(N_SAMPLES, SEED)
    results = run_reference_checks(frame)
    shape = run_shape_checks(frame)
    independent = run_independent_check(frame, results)
    timing_trials, timing_summary = benchmark_integration_orders(frame)

    feller_analysis = results.groupby("feller_satisfied", as_index=False).agg(
        n=("sample_id", "count"),
        valid_price=("valid_price", "sum"),
        converged_64_128 = ("converged_64_128", "sum"), 
        mean_difference_64_128=("difference_64_128", "mean"),
        max_difference_64_128=("difference_64_128", "max"),
    )

    results.to_csv(OUTPUT/ "reference_validation.csv", index=False)
    shape.to_csv(OUTPUT/ "strike_shape_checks.csv", index=False)
    feller_analysis.to_csv(OUTPUT/ "feller_analysis.csv", index=False)
    independent.to_csv(OUTPUT/ "independent_fourier_comparison_analysis.csv", index=False)
    timing_trials.to_csv(OUTPUT/ "integration_timing_trials.csv", index=False)
    timing_summary.to_csv(OUTPUT/ "integration_timing_summary.csv", index=False)
    plot_convergence(results)
    plot_benchmark(timing_summary)

    n_independent_ok = int(independent["abs_difference"].notna().sum())
    n_shape = len(shape)
    ind_max = independent["abs_difference"].max()
    summary = {
        "parameters_combinations": len(results),
        "valid_prices": int(results["valid_price"].sum()),
        "successful_prices_by_order":{
            str(o): int(results[f"price_{o}"].notna().sum())
            for o in INTEGRATION_ORDERS
        }, 
        "converged_32_64": int(results["converged_32_64"].sum()),
        "converged_64_128": int(results["converged_64_128"].sum()),
        "mean_difference_64_128": float(results["difference_64_128"].mean()),
        "max_difference_32_64": float(results["difference_32_64"].max()),
        "max_difference_64_128": float(results["difference_64_128"].max()),
        "feller_satisfied": int(results["feller_satisfied"].sum()),
        "strike_curve_checks": n_shape, 
        "strike_monotonicity_passed": int(shape["strike_monotonicity_ok"].fillna(False).sum()),
        "strike_convexity_passed": int(shape["strike_convexity_ok"].fillna(False).sum()),
        "put_call_parity_passed": int(shape["put_call_parity_ok"].fillna(False).sum()),
        "max_put_call_parity_error": (
            float(shape["put_call_parity_abs_error"].max())
            if shape["put_call_parity_abs_error"].notna().any()
            else None
        ),
        "independent_solver_method": "SciPy Carr-Madan Fourier integral", 
        "independent_cases_attempted": len(independent),
        "independent_cases_priced": n_independent_ok,
        "independent_cases_within_tolerance": int(independent["within_tolerance"].sum()),
        "independent_max_difference": (float(ind_max) if pd.notna(ind_max) else None), 
        "independent_solver_comparison_completed": bool(n_independent_ok == len(independent) and len(independent) > 0),
        "independent_all_within_tolerance": bool(independent["within_tolerance"].all()),
        "benchmark_repeats": BENCHMARK_REPEAT,
        "timing_median_ms_per_option":{
            str(row.integration_order):float(row.median_ms_per_option)
            for row in timing_summary.itertuples(index=False)
        }, 
        "tolerances":{
            "convergence": CONVERGENCE_TOL,
            "independent": INDEPENDENT_TOL, 
            "financial_bounds": BOUND_TOL, 
            "strike_shape": SHAPE_TOL,
            "put_call_parity": PARITY_TOL,
        }, 
        "reproducibility": {
            "seed" : SEED, 
            "valuation_date": "2026-01-02",
            "day_count": "Actual/365 Fixed", 
            "quantlib_version": str(getattr(ql, "__version__", "unknown")),
            "scipy_version": scipy.__version__, 
            "python_version": platform.python_version(),
        },
    }
    (OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\n"+"="*64)
    print("EXPERIMENT: HESTON REFERENCE VALIDATION")
    print("="*64)
    print(f"Valid Prices:               {summary['valid_prices']}/{N_SAMPLES}")
    print(f"Converged (32 vs 64):       {summary['converged_32_64']}/{N_SAMPLES}")
    print(f"Converged (64 vs 128):      {summary['converged_64_128']}/{N_SAMPLES}"),
    print(f"Max difference (32/64):     {summary['max_difference_32_64']:.3e}")
    print(f"Max difference (64/128):    {summary['max_difference_64_128']:.3e}")
    print(f"Feller satisfied:           {summary['feller_satisfied']}/{N_SAMPLES}")
    print(f"Monotonic Strike Curves:    {summary['strike_monotonicity_passed']}/{n_shape}")
    print(f"Convex strike curves:       {summary['strike_convexity_passed']}/{n_shape}")
    print(f"Put-call parity passed:     {summary['put_call_parity_passed']}/{n_shape}")
    print(f"Independent Fourier Cases:  {n_independent_ok}/{len(independent)} priced")
    print(f"Independent Within Tol:     {summary['independent_cases_within_tolerance']}/{len(independent)}")
    if summary["independent_max_difference"] is not None: 
        print(f"Independent max diff:   {summary['independent_max_difference']:.3e}")
    print("\n Median milliseconds per option (repeat benchmarks):")
    print(timing_summary.to_string(index=False))
    failed = independent[(independent["error"] != "")|
                         (independent["fourier_warning"] != "")|
                         ~independent["within_tolerance"]]
    if len(failed):
        print("\n Independent cross-check items to investigate:")
        print(failed[["sample_id", "abs_difference", "fourier_warning", "error"]]
              .to_string(index=False))
    print(f"\nSaved results to: {OUTPUT}")


if __name__ == "__main__":
    main()



Reference pricing: 20/100 parameter combinations
Reference pricing: 40/100 parameter combinations
Reference pricing: 60/100 parameter combinations
Reference pricing: 80/100 parameter combinations
Reference pricing: 100/100 parameter combinations

EXPERIMENT: HESTON REFERENCE VALIDATION
Valid Prices:               100/100
Converged (32 vs 64):       100/100
Converged (64 vs 128):      100/100
Max difference (32/64):     1.641e-10
Max difference (64/128):    2.720e-15
Feller satisfied:           85/100
Monotonic Strike Curves:    100/100
Convex strike curves:       100/100
Put-call parity passed:     100/100
Independent Fourier Cases:  24/24 priced
Independent Within Tol:     24/24
Independent max diff:   2.018e-13

 Median milliseconds per option (repeat benchmarks):
 integration_order  median_ms_per_option  min_ms_per_option  max_ms_per_option
                32              0.176046           0.172414           0.199223
                64              0.389695           0.380258      

## extended validation 
1000 independent Latin-Hypercube cases

In [43]:
EXTENDED_N = 1000 
EXTENDED_SEED = 123 

extended_frame = sample_liu_params(EXTENDED_N, EXTENDED_SEED)
extended_results = run_reference_checks(extended_frame)

for order in INTEGRATION_ORDERS:
    prices = pd.to_numeric( 
        extended_results[f"price_{order}"],
        errors="coerce"
    )
    extended_results[f"finite_{order}"] = np.isfinite(prices)

extended_results["all_prices_finite"] = extended_results[
    [f"finite_{order}" for order in INTEGRATION_ORDERS]
].all(axis=1)
extended_results["all_checks_passed"] = (
    extended_results["all_prices_finite"]
    & extended_results["valid_price"]
    & extended_results["converged_32_64"]
    & extended_results["converged_64_128"]
)

extended_results.to_csv(OUTPUT / "extended_validation_1000.csv", index=False)
extended_summary = {
    "seed": EXTENDED_SEED,
    "n_cases": len(extended_results), 
    "all_prices_finite": int(extended_results["all_prices_finite"].sum()),
    "valid_reference_prices": int(extended_results["valid_price"].sum()), 
    "converged_32_64": int(extended_results["converged_32_64"].sum()), 
    "converged_64_128": int(extended_results["converged_64_128"].sum()), 
    "max_difference_32_64": float(extended_results["difference_32_64"].max()), 
    "max_difference_64_128": float(extended_results["difference_64_128"].max()),
    "feller_satisfied": int(extended_results["feller_satisfied"].sum()), 
    "feller_violated": int((~extended_results["feller_satisfied"]).sum()),
    "all_checks_passed": int(extended_results["all_checks_passed"].sum()),
    "convergence_tolerance": CONVERGENCE_TOL,
}

(OUTPUT/"extended_summary_1000.json").write_text(
    json.dumps(extended_summary, indent=2), encoding="utf-8"
)

print("\nEXTENDED VALIDATION (independent LHS sample)")
for key, value in extended_summary.items():
    print(f"{key}: {value}")

failed_cases = extended_results.loc[~extended_results["all_checks_passed"]]
if not failed_cases.empty:
    print("\nCases requiring investigation:")
    display(failed_cases[[
        "sample_id", "m", "T", "rho", "kappa", "theta", "sigma", "v0", "difference_32_64", "difference_64_128", "valid_price", "error_32", "error_64", "error_128"
    ]])

Reference pricing: 20/1000 parameter combinations
Reference pricing: 40/1000 parameter combinations
Reference pricing: 60/1000 parameter combinations
Reference pricing: 80/1000 parameter combinations
Reference pricing: 100/1000 parameter combinations
Reference pricing: 120/1000 parameter combinations
Reference pricing: 140/1000 parameter combinations
Reference pricing: 160/1000 parameter combinations
Reference pricing: 180/1000 parameter combinations
Reference pricing: 200/1000 parameter combinations
Reference pricing: 220/1000 parameter combinations
Reference pricing: 240/1000 parameter combinations
Reference pricing: 260/1000 parameter combinations
Reference pricing: 280/1000 parameter combinations
Reference pricing: 300/1000 parameter combinations
Reference pricing: 320/1000 parameter combinations
Reference pricing: 340/1000 parameter combinations
Reference pricing: 360/1000 parameter combinations
Reference pricing: 380/1000 parameter combinations
Reference pricing: 400/1000 paramet